# КИМ 5.1. Рекуррентные сети и 1D-CNN для последовательностей

**Модуль 5. Нейронные сети для анализа последовательностей** · Курс «Основы нейронных сетей» · УрФУ

Проверяемые результаты (КРМ v3.0):
- **DL-1.5 (Б):** применяет LSTM и GRU, выбирает hidden size и сравнивает с
  1D-CNN.
- **DL-4.1 (Б):** применяет regex, токенизацию, морфологический и синтаксический
  анализ, формирует словарь текста.

Подробное описание, критерии и контрольные вопросы — в `kim-01-sequences.md`.

---
## Часть А. Предобработка текстов

Работаем с задачей бинарной классификации тональности отзывов. Используем
`pymorphy3` (лемматизация) и `nltk` (токенизация). Данные —
[`ai-forever/ru-reviews-classification`](https://huggingface.co/datasets/ai-forever/ru-reviews-classification),
в метаданных указана Apache-2.0, но происхождение текстов не описано. Используется
только опубликованный сплит `train` в revision
`0f67d914f396ce22917dc6463ec619799b3b08d2`; копия данных не включается в
репозиторий, нейтральные отзывы исключаются.

### 0. Импорт библиотек

In [ ]:
# < ENTER YOUR CODE HERE >

### 1. Regex-нормализация и тесты

Реализуйте `normalize_markup(text)`: замените URL, e-mail и числа устойчивыми
маркерами, нормализуйте пробелы. Добавьте минимум пять `assert` для обычных и
граничных случаев.

In [ ]:
# < ENTER YOUR CODE HERE >  # normalize_markup + 5 assert

### 2. Токенизация, лемматизация, удаление стоп-слов

Реализуйте функцию `preprocess(text)`: приведение к нижнему регистру, токенизация
(`nltk.word_tokenize`), удаление пунктуации и стоп-слов, лемматизация
(`pymorphy3.MorphAnalyzer`). Для модельного конвейера задайте фиксированный
встроенный список русских стоп-слов: наличие локального корпуса NLTK не должно
менять последовательности и результаты.

In [ ]:
# < ENTER YOUR CODE HERE >  # preprocess

### 3. Синтаксический разбор

Через Natasha, spaCy или Stanza разберите минимум пять предложений. Выведите
таблицу `token — head — dependency` и письменно объясните две связи.

In [ ]:
# < ENTER YOUR CODE HERE >  # синтаксический разбор 5 предложений + таблица

### 4. Загрузка данных и построение словаря

Загрузите закреплённый `train.jsonl` (revision
`0f67d914f396ce22917dc6463ec619799b3b08d2`, SHA-256
`0b97698a0c6871437d17e07c973018af9b8c9230ec9048d85cb875cc2c2470ea`).
Проверяйте существующий кэш; новый файл сначала скачивайте во временный файл,
проверяйте и только затем заменяйте кэш атомарно. Путь к `attachments/data`
определяйте как для запуска из каталога ноутбука, так и из корня репозитория.

Оставьте метки negative/positive, преобразуйте их в `0`/`1` и при `seed=42`
возьмите по 500 объектов каждого класса **только из опубликованного `train`**.
Сделайте локальное стратифицированное разбиение 640/160/200 на
train/validation/test. Локальный test не является официальным test-сплитом
набора. Постройте словарь **только по локальному train** (`max_words=10000`) со
спецкодами: `0` — паддинг, `1` — неизвестное слово (OOV), нумерация слов с `2`.

In [ ]:
# < ENTER YOUR CODE HERE >  # verified pinned train + fixed subset/split + train-only vocab

### 5. Преобразование текстов в последовательности

Преобразуйте тексты в последовательности индексов и выровняйте длину
(`pad_sequences(maxlen=200)`).

In [ ]:
# < ENTER YOUR CODE HERE >  # text_to_sequence + pad_sequences

---
## Часть Б. Классификация текстов: LSTM vs GRU vs 1D-CNN

### 6. LSTM и GRU с выбором hidden size

Обучите LSTM и GRU в одинаковом протоколе. Для каждого типа сравните
`hidden_size=32` и `hidden_size=64`. Выберите конфигурацию по validation,
зафиксируйте число параметров, точность и время.

In [ ]:
# < ENTER YOUR CODE HERE >  # LSTM/GRU × hidden_size 32/64 + validation selection

### 7. 1D-CNN-модель

Соберите альтернативу: `Embedding` → `Conv1D(250, 5, relu)` → `GlobalMaxPooling1D`
→ `Dense(128, relu)` → `Dropout(0.2)` → `Dense(1, sigmoid)`.

In [ ]:
# < ENTER YOUR CODE HERE >  # Sequential с Conv1D

### 8. Сравнение LSTM, GRU и 1D-CNN

Сравните лучшие LSTM и GRU с 1D-CNN по точности, времени и числу параметров.
Постройте столбчатые графики validation accuracy и времени. Выберите архитектуру
только по validation; test используйте один раз для итоговой оценки и не меняйте
после него гиперпараметры.

In [ ]:
# < ENTER YOUR CODE HERE >  # comparison table + validation accuracy/time charts + conclusion

**Контрольный вопрос:** что извлекает одномерная свёртка над
эмбеддингами? Зачем `GlobalMaxPooling1D`? Сравните с тем, как работает LSTM.

> *Ваш ответ:* ...

---
## Часть В. Прогнозирование временного ряда (Jena Climate)

Прогноз температуры по данным метеостанции Йены (~420 000 измерений, 14 признаков).

### 9. Загрузка и подготовка данных

In [ ]:
# < ENTER YOUR CODE HERE >  # load CSV + нормализация по train

Нормализуйте признаки по статистикам **обучающей** выборки. Сформируйте
окна через `keras.utils.timeseries_dataset_from_array` или эквивалентный
`torch.utils.data.Dataset` с `sequence_length=120`, `sampling_rate=6`.

### 10. Наивный бейзлайн

Реализуйте `evaluate_naive_method`: прогноз последним известным значением.
Посчитайте MAE бейзлайна — с ним будем сравнивать нейросеть.

In [ ]:
# < ENTER YOUR CODE HERE >  # naive baseline

### 11. LSTM-модель для временного ряда

Соберите простую сеть: `LSTM(16)` → `Dense(1)`. Обучите, оцените MAE. Сравните
с бейзлайном.

In [ ]:
# < ENTER YOUR CODE HERE >  # LSTM для timeseries

### 12. Обязательное усложнение: recurrent dropout

Обучите `LSTM(32, recurrent_dropout=0.25)`. В PyTorch встроенный аргумент
`dropout` у `nn.LSTM` не эквивалентен recurrent dropout: реализуйте проход через
`LSTMCell`, применяя одну и ту же dropout-маску к `h[t-1]` на всех шагах
последовательности. Stacked GRU можно добавить как необязательное расширение.

В итоговой таблице обязательны наивный бейзлайн, `LSTM(16)` и
`LSTM(32, recurrent_dropout=0.25)` с validation/test MAE, числом параметров и
временем обучения.

In [ ]:
# < ENTER YOUR CODE HERE >  # LSTMCell recurrent_dropout=0.25 + required comparison table

**Контрольный вопрос:** чем задача регрессии временного ряда отличается
от классификации (функция потерь, выходной слой, метрики)? Зачем нужен наивный
бейзлайн?

> *Ваш ответ:* ...

---
## Итог

Перед сдачей убедитесь, что:
- [ ] regex проходит 5 тестов, сохранён синтаксический разбор 5 предложений;
- [ ] обучены LSTM и GRU с hidden size 32/64 и 1D-CNN, сделано сравнение;
- [ ] для временного ряда сравнены бейзлайн, LSTM(16) и LSTM(32) с recurrent
      dropout 0.25;
- [ ] вы готовы объяснить затухание градиента в RNN и механизм LSTM/GRU.

**Формат сдачи:** заполненный ноутбук + устная защита.